[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/44.%20The%20Carbon%20Footprint%20Modeling%20Problem/P44-Tier-2_executed.ipynb)

---


# 44. The Carbon Footprint Modeling Problem

## Tier 2: The Classic Heuristic — Sweep Algorithm

### Goal and workflow

This notebook implements a carbon-aware version of the sweep heuristic for a capacitated vehicle-routing problem. It orders customers by their polar angle around a depot, forms routes from consecutive customers in that order, and assigns each route the feasible vehicle type with the lowest emissions per kilometre.

The workflow is deliberately sequential:

1. Represent the depot and customers in a two-dimensional coordinate plane.
2. Calculate each customer's distance and polar angle relative to the depot.
3. Sort customers by angle.
4. Sweep through that order, closing a route when adding the next customer would exceed the largest vehicle capacity.
5. For each completed route, select the feasible vehicle with the lowest emissions per kilometre.
6. Calculate route distance, emissions, and operating cost, then inspect and visualize the result.

### Background: polar coordinates and the sweep idea

A customer at Cartesian coordinates \((x_i,y_i)\), relative to a depot at \((x_0,y_0)\), has displacement \((x_i-x_0,y_i-y_0)\). Its distance from the depot is the Euclidean distance, and its angle describes its direction from the positive horizontal axis. Sorting by angle creates a circular order around the depot. The sweep heuristic uses this order to group nearby directions into consecutive route segments; the grouping is a construction rule, not a guarantee of globally optimal routes.

### Mathematical model used by this implementation

Let \(C\) be the set of customers and \(V\) the set of vehicle types. Customer \(i\in C\) has coordinates \((x_i,y_i)\) and demand \(q_i\). The depot is at \((x_0,y_0)\). Vehicle type \(v\in V\) has capacity \(Q_v\), emissions factor \(e_v\) in kg CO\(_2\)e per kilometre, and operating cost \(c_v\) per kilometre.

The polar coordinates used to order customers are

$$r_i=\sqrt{(x_i-x_0)^2+(y_i-y_0)^2},\qquad
\theta_i=\operatorname{atan2}(y_i-y_0,x_i-x_0)\pmod{360^\circ}.$$

The implementation sorts customers in ascending \(\theta_i\). Starting from the first customer in that order, it adds customers to the current route while the total demand does not exceed the largest vehicle capacity, \(Q_{\max}=\max_{v\in V}Q_v\). If the next customer would exceed that capacity, the current route is closed and a new one begins. Thus every route is a consecutive block in the sorted list.

For a route containing customers \((i_1,\ldots,i_k)\) in sweep order, the route starts and ends at the depot. Its distance is

$$D(i_1,\ldots,i_k)=d_{0i_1}+\sum_{j=1}^{k-1}d_{i_ji_{j+1}}+d_{i_k0},$$

where \(d_{ab}=\sqrt{(x_a-x_b)^2+(y_a-y_b)^2}\). The route demand is \(L=\sum_{j=1}^{k}q_{i_j}\). Its vehicle is chosen from the capacity-feasible set \(V(L)=\{v\in V:Q_v\ge L\}\) by minimizing the emissions factor:

$$v^*\in\underset{v\in V(L)}{\operatorname{argmin}}\ e_v.$$

The emissions and operating cost of that route are \(D e_{v^*}\) and \(D c_{v^*}\), respectively. The total carbon footprint of the constructed solution is the sum of route emissions. This is a heuristic solution: the sweep order and capacity-based grouping are fixed construction rules, and the vehicle choice is made separately for each constructed route. The method does not solve a global route-optimization problem and does not provide an optimality guarantee.

### Scope and modeling assumptions

- Coordinates are treated as planar distances in kilometres, so the Euclidean distance formula produces kilometres. Demand is measured in abstract demand units.
- Each route starts and ends at the single depot, and each customer is served exactly once.
- A route follows the ascending polar-angle order of its customers; no route reversal or within-route reordering is performed.
- Vehicle availability is unlimited by type. The only vehicle constraint is route capacity.
- Emissions and cost are linear in distance and depend only on vehicle type. Speed, congestion, load-dependent fuel use, time windows, and other operational effects are outside this model.
- If more than one feasible vehicle type has the same minimum emissions factor, the first such type in the input list is selected.

### Concrete example

The example contains six customers around a depot at \((0,0)\). Each customer has demand 15 units. The Truck has capacity 100 units and emits 0.8 kg CO\(_2\)e/km; the Van has capacity 100 units and emits 0.6 kg CO\(_2\)e/km. Both vehicle types can carry the total demand of all six customers, so the sweep's capacity rule does not require a route split for this data. The actual angle order and route metrics are calculated from the coordinates in the code; they are not assumed in advance.

### Interpreting the results

The route listing shows the angular order, capacity-based route boundaries, and selected vehicle for each route. The emissions calculation follows directly from route distance multiplied by the selected vehicle's emissions factor. Since this notebook does not calculate an exact optimum or use an independent benchmark, its output should be interpreted as the result of the specified heuristic—not as an optimal solution or as a measured optimality gap.

In [ ]:
import math
from dataclasses import dataclass
from typing import List

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("default")

In [ ]:
@dataclass
class Customer:
    """A customer location, demand, and depot-relative polar coordinates."""

    id: str
    x: float
    y: float
    demand: float
    angle: float = 0.0
    distance: float = 0.0


@dataclass
class Vehicle:
    """A vehicle type with capacity, emissions, and operating-cost rates."""

    id: str
    vehicle_type: str
    capacity: float
    emission_factor: float
    cost_per_km: float


@dataclass
class Route:
    """A depot-to-depot route and its calculated performance measures."""

    id: str
    customers: List[str]
    vehicle: Vehicle
    total_demand: float
    total_distance: float
    total_emissions: float
    total_cost: float


@dataclass
class SweepProblem:
    """Input data for the single-depot sweep heuristic."""

    depot_x: float
    depot_y: float
    customers: List[Customer]
    vehicles: List[Vehicle]

In [ ]:
def create_concrete_example() -> SweepProblem:
    """Create the six-customer example and its two vehicle types."""
    depot_x, depot_y = 0.0, 0.0

    customers = [
        Customer("C1", 8.5, 6.0, 15.0),
        Customer("C2", 9.0, 4.5, 15.0),
        Customer("C3", 8.5, 2.0, 15.0),
        Customer("C4", 4.5, 2.5, 15.0),
        Customer("C5", 3.0, 4.0, 15.0),
        Customer("C6", 4.0, 6.5, 15.0),
    ]

    vehicles = [
        Vehicle("Truck", "Truck", 100.0, 0.8, 1.2),
        Vehicle("Van", "Van", 100.0, 0.6, 1.0),
    ]

    return SweepProblem(depot_x, depot_y, customers, vehicles)


problem = create_concrete_example()
print(
    f"Problem created with {len(problem.customers)} customers "
    f"and {len(problem.vehicles)} vehicle types."
)
print(f"Depot location: ({problem.depot_x:.1f}, {problem.depot_y:.1f}) km")

In [ ]:
def calculate_polar_coordinates(problem: SweepProblem) -> None:
    """Store each customer's depot distance and angle in degrees."""
    for customer in problem.customers:
        dx = customer.x - problem.depot_x
        dy = customer.y - problem.depot_y
        customer.distance = math.hypot(dx, dy)

        angle_degrees = math.degrees(math.atan2(dy, dx))
        customer.angle = angle_degrees % 360.0

    print("Customer polar coordinates (distance in km, angle in degrees):")
    print(f"{'Customer':<10} {'X':>8} {'Y':>8} {'Distance':>12} {'Angle':>10}")
    print("-" * 52)
    for customer in problem.customers:
        print(
            f"{customer.id:<10} {customer.x:>8.1f} {customer.y:>8.1f} "
            f"{customer.distance:>12.2f} {customer.angle:>9.1f}°"
        )


calculate_polar_coordinates(problem)

In [ ]:
def sort_customers_by_angle(problem: SweepProblem) -> List[Customer]:
    """Return customers in ascending depot-relative polar-angle order."""
    sorted_customers = sorted(problem.customers, key=lambda customer: customer.angle)

    print("\nCustomers sorted by polar angle:")
    print(f"{'Order':<8} {'Customer':<10} {'Angle':>10} {'Distance (km)':>16}")
    print("-" * 48)
    for index, customer in enumerate(sorted_customers, start=1):
        print(
            f"{index:<8} {customer.id:<10} {customer.angle:>9.1f}° "
            f"{customer.distance:>16.2f}"
        )

    return sorted_customers


sorted_customers = sort_customers_by_angle(problem)

In [ ]:
def calculate_route_distance(
    route_customers: List[Customer],
    depot_x: float,
    depot_y: float,
) -> float:
    """Calculate Euclidean distance for a depot-to-depot customer sequence."""
    if not route_customers:
        return 0.0

    total_distance = math.hypot(
        route_customers[0].x - depot_x,
        route_customers[0].y - depot_y,
    )

    for current_customer, next_customer in zip(
        route_customers, route_customers[1:]
    ):
        total_distance += math.hypot(
            next_customer.x - current_customer.x,
            next_customer.y - current_customer.y,
        )

    total_distance += math.hypot(
        route_customers[-1].x - depot_x,
        route_customers[-1].y - depot_y,
    )
    return total_distance


def select_best_vehicle(
    route_distance: float,
    total_demand: float,
    vehicles: List[Vehicle],
) -> Vehicle:
    """Choose the capacity-feasible vehicle with minimum route emissions."""
    if not vehicles:
        raise ValueError("At least one vehicle type is required.")

    feasible_vehicles = [
        vehicle
        for vehicle in vehicles
        if vehicle.capacity >= total_demand
    ]
    if not feasible_vehicles:
        raise ValueError(
            f"No vehicle can carry the route demand of {total_demand:g} units."
        )

    # Every candidate serves the same route distance, so minimizing this
    # product is equivalent to minimizing the vehicle's emissions factor.
    return min(
        feasible_vehicles,
        key=lambda vehicle: route_distance * vehicle.emission_factor,
    )


print("Route-distance and vehicle-selection functions are ready.")

In [ ]:
def carbon_aware_sweep_algorithm(problem: SweepProblem) -> List[Route]:
    """Build capacity-feasible routes in angular order and assign vehicles."""
    if not problem.vehicles:
        raise ValueError("At least one vehicle type is required.")
    if any(customer.demand < 0 for customer in problem.customers):
        raise ValueError("Customer demands must be non-negative.")

    max_capacity = max(vehicle.capacity for vehicle in problem.vehicles)
    if max_capacity < 0:
        raise ValueError("Vehicle capacities must be non-negative.")
    if any(customer.demand > max_capacity for customer in problem.customers):
        raise ValueError(
            "At least one customer exceeds the capacity of every vehicle type."
        )

    print("=" * 64)
    print("CARBON-AWARE SWEEP ALGORITHM")
    print("=" * 64)

    # The sweep uses the depot-relative angles calculated before this call.
    ordered_customers = sort_customers_by_angle(problem)
    routes: List[Route] = []
    current_route_customers: List[Customer] = []
    current_demand = 0.0
    route_id = 1

    print("\n--- Route construction ---")
    for customer in ordered_customers:
        if current_route_customers and (
            current_demand + customer.demand > max_capacity
        ):
            route_distance = calculate_route_distance(
                current_route_customers,
                problem.depot_x,
                problem.depot_y,
            )
            best_vehicle = select_best_vehicle(
                route_distance,
                current_demand,
                problem.vehicles,
            )
            route = Route(
                id=f"R{route_id}",
                customers=[item.id for item in current_route_customers],
                vehicle=best_vehicle,
                total_demand=current_demand,
                total_distance=route_distance,
                total_emissions=route_distance * best_vehicle.emission_factor,
                total_cost=route_distance * best_vehicle.cost_per_km,
            )
            routes.append(route)
            print(
                f"{route.id}: {' → '.join(route.customers)} | "
                f"Demand: {route.total_demand:g} units | "
                f"Distance: {route.total_distance:.2f} km | "
                f"Vehicle: {route.vehicle.vehicle_type} | "
                f"Emissions: {route.total_emissions:.2f} kg CO₂e"
            )

            route_id += 1
            current_route_customers = []
            current_demand = 0.0

        current_route_customers.append(customer)
        current_demand += customer.demand
        print(
            f"Added {customer.id} at {customer.angle:.1f}°; "
            f"current route demand is {current_demand:g} units."
        )

    if current_route_customers:
        route_distance = calculate_route_distance(
            current_route_customers,
            problem.depot_x,
            problem.depot_y,
        )
        best_vehicle = select_best_vehicle(
            route_distance,
            current_demand,
            problem.vehicles,
        )
        route = Route(
            id=f"R{route_id}",
            customers=[item.id for item in current_route_customers],
            vehicle=best_vehicle,
            total_demand=current_demand,
            total_distance=route_distance,
            total_emissions=route_distance * best_vehicle.emission_factor,
            total_cost=route_distance * best_vehicle.cost_per_km,
        )
        routes.append(route)
        print(
            f"{route.id}: {' → '.join(route.customers)} | "
            f"Demand: {route.total_demand:g} units | "
            f"Distance: {route.total_distance:.2f} km | "
            f"Vehicle: {route.vehicle.vehicle_type} | "
            f"Emissions: {route.total_emissions:.2f} kg CO₂e"
        )

    assigned_customers = [customer_id for route in routes for customer_id in route.customers]
    expected_customers = [customer.id for customer in problem.customers]
    assert len(assigned_customers) == len(set(assigned_customers))
    assert set(assigned_customers) == set(expected_customers)
    for route in routes:
        assert route.total_demand <= route.vehicle.capacity
        assert math.isclose(
            route.total_emissions,
            route.total_distance * route.vehicle.emission_factor,
            rel_tol=1e-9,
            abs_tol=1e-9,
        )

    return routes


routes = carbon_aware_sweep_algorithm(problem)

In [ ]:
def analyze_sweep_solution(routes: List[Route]) -> None:
    """Report route-level and aggregate measures for the constructed solution."""
    total_emissions = sum(route.total_emissions for route in routes)
    total_distance = sum(route.total_distance for route in routes)
    total_cost = sum(route.total_cost for route in routes)
    total_demand = sum(route.total_demand for route in routes)
    average_emissions_per_km = (
        total_emissions / total_distance if total_distance > 0 else 0.0
    )

    print("\n" + "=" * 64)
    print("SWEEP SOLUTION ANALYSIS")
    print("=" * 64)
    print("\n--- Aggregate measures ---")
    print(f"Number of routes: {len(routes)}")
    print(f"Total distance: {total_distance:.2f} km")
    print(f"Total emissions: {total_emissions:.2f} kg CO₂e")
    print(f"Total operating cost: ${total_cost:.2f}")
    print(f"Total demand served: {total_demand:g} units")
    print(
        "Average emissions per kilometre: "
        f"{average_emissions_per_km:.3f} kg CO₂e/km"
    )

    print("\n--- Route details ---")
    print(
        f"{'Route':<8} {'Customers':<28} {'Vehicle':<10} "
        f"{'Demand':>10} {'Distance (km)':>15} {'Emissions (kg CO₂e)':>22}"
    )
    print("-" * 96)
    for route in routes:
        customer_sequence = " → ".join(route.customers)
        print(
            f"{route.id:<8} {customer_sequence:<28} "
            f"{route.vehicle.vehicle_type:<10} {route.total_demand:>10.1f} "
            f"{route.total_distance:>15.2f} {route.total_emissions:>22.2f}"
        )

    vehicle_usage = {}
    for route in routes:
        vehicle_type = route.vehicle.vehicle_type
        usage = vehicle_usage.setdefault(
            vehicle_type,
            {"routes": 0, "emissions": 0.0, "distance": 0.0},
        )
        usage["routes"] += 1
        usage["emissions"] += route.total_emissions
        usage["distance"] += route.total_distance

    print("\n--- Vehicle use ---")
    for vehicle_type, usage in vehicle_usage.items():
        distance = usage["distance"]
        rate = usage["emissions"] / distance if distance > 0 else 0.0
        print(
            f"{vehicle_type}: {usage['routes']} route(s), "
            f"{usage['emissions']:.2f} kg CO₂e, "
            f"{rate:.3f} kg CO₂e/km"
        )


analyze_sweep_solution(routes)

In [ ]:
def visualize_sweep_solution(
    problem: SweepProblem,
    routes: List[Route],
) -> None:
    """Plot route geometry, angular assignments, emissions, and totals."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle("Carbon-Aware Sweep Heuristic", fontsize=16, fontweight="bold")

    route_colors = ["tab:green", "tab:orange", "tab:purple", "tab:brown", "tab:pink", "tab:gray"]
    color_by_customer = {
        customer_id: route_colors[index % len(route_colors)]
        for index, route in enumerate(routes)
        for customer_id in route.customers
    }
    customer_by_id = {customer.id: customer for customer in problem.customers}

    ax1 = axes[0, 0]
    ax1.scatter(
        problem.depot_x,
        problem.depot_y,
        color="red",
        marker="*",
        s=250,
        edgecolors="black",
        label="Depot",
        zorder=5,
    )
    ax1.annotate(
        "Depot",
        (problem.depot_x, problem.depot_y),
        xytext=(6, 6),
        textcoords="offset points",
        fontweight="bold",
    )
    for customer in problem.customers:
        ax1.scatter(
            customer.x,
            customer.y,
            color=color_by_customer.get(customer.id, "tab:blue"),
            s=75,
            edgecolors="black",
            zorder=3,
        )
        ax1.annotate(
            f"{customer.id} ({customer.angle:.0f}°)",
            (customer.x, customer.y),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )

    for index, route in enumerate(routes):
        color = route_colors[index % len(route_colors)]
        route_customers = [customer_by_id[item] for item in route.customers]
        if not route_customers:
            continue
        x_values = [problem.depot_x] + [item.x for item in route_customers] + [problem.depot_x]
        y_values = [problem.depot_y] + [item.y for item in route_customers] + [problem.depot_y]
        ax1.plot(
            x_values,
            y_values,
            color=color,
            linewidth=2,
            alpha=0.8,
            label=f"{route.id} ({route.vehicle.vehicle_type})",
        )

    ax1.set_xlabel("X coordinate (km)")
    ax1.set_ylabel("Y coordinate (km)")
    ax1.set_title("Depot-to-depot route geometry")
    ax1.grid(True, alpha=0.3)
    ax1.axis("equal")
    ax1.legend(loc="best")

    fig.delaxes(axes[0, 1])
    ax2 = fig.add_subplot(2, 2, 2, projection="polar")
    for index, route in enumerate(routes):
        route_customers = [customer_by_id[item] for item in route.customers]
        if route_customers:
            ax2.scatter(
                [math.radians(item.angle) for item in route_customers],
                [item.distance for item in route_customers],
                color=route_colors[index % len(route_colors)],
                s=75,
                edgecolors="black",
                label=f"{route.id} ({route.vehicle.vehicle_type})",
            )
            for customer in route_customers:
                ax2.annotate(
                    customer.id,
                    (math.radians(customer.angle), customer.distance),
                    xytext=(4, 4),
                    textcoords="offset points",
                    fontsize=8,
                )
    ax2.set_theta_zero_location("E")
    ax2.set_title("Customer angles and depot distances")
    if routes:
        ax2.legend(loc="best", bbox_to_anchor=(1.25, 1.1))

    ax3 = axes[1, 0]
    route_labels = [route.id for route in routes]
    emissions = [route.total_emissions for route in routes]
    bars = ax3.bar(
        route_labels,
        emissions,
        color=[route_colors[index % len(route_colors)] for index in range(len(routes))],
        edgecolor="black",
        alpha=0.8,
    )
    ax3.set_xlabel("Route")
    ax3.set_ylabel("Emissions (kg CO₂e)")
    ax3.set_title("Route emissions and assigned vehicle")
    ax3.grid(True, alpha=0.3, axis="y")
    for bar, route in zip(bars, routes):
        ax3.annotate(
            f"{bar.get_height():.2f}\n{route.vehicle.vehicle_type}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    ax4 = axes[1, 1]
    total_distance = sum(route.total_distance for route in routes)
    total_emissions = sum(route.total_emissions for route in routes)
    total_cost = sum(route.total_cost for route in routes)
    total_demand = sum(route.total_demand for route in routes)
    average_rate = total_emissions / total_distance if total_distance > 0 else 0.0
    summary_lines = [
        f"Routes: {len(routes)}",
        f"Customers served: {sum(len(route.customers) for route in routes)}",
        f"Demand served: {total_demand:g} units",
        f"Total distance: {total_distance:.2f} km",
        f"Total emissions: {total_emissions:.2f} kg CO₂e",
        f"Operating cost: ${total_cost:.2f}",
        f"Average emissions rate: {average_rate:.3f} kg CO₂e/km",
    ]
    ax4.axis("off")
    ax4.set_title("Aggregate solution measures")
    ax4.text(
        0.05,
        0.85,
        "\n\n".join(summary_lines),
        transform=ax4.transAxes,
        va="top",
        fontsize=11,
        linespacing=1.4,
    )

    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()


visualize_sweep_solution(problem, routes)

In [ ]:
def performance_analysis(problem: SweepProblem, routes: List[Route]) -> None:
    """Explain asymptotic resource use and the heuristic's limitations."""
    n_customers = len(problem.customers)
    n_vehicle_types = len(problem.vehicles)
    n_routes = len(routes)

    print("=" * 64)
    print("ALGORITHM PERFORMANCE ANALYSIS")
    print("=" * 64)
    print("\n--- Time complexity ---")
    print(f"Customers (n): {n_customers}")
    print(f"Vehicle types (v): {n_vehicle_types}")
    print("Angular sorting: O(n log n)")
    print("Sweep construction and route-distance calculations: O(n) total")
    print("Vehicle selection: O(v) per route, or O(r v) for r routes")
    print("Overall: O(n log n + n + r v), with r ≤ n")
    print("For a fixed number of vehicle types, this is O(n log n).")

    print("\n--- Space complexity ---")
    print("The sorted customer list and route records require O(n) space.")
    print("This is auxiliary and output storage, in addition to the input data.")

    total_emissions = sum(route.total_emissions for route in routes)
    print("\n--- Solution interpretation ---")
    print(f"Constructed solution emissions: {total_emissions:.2f} kg CO₂e")
    print(
        "This value describes the routes generated by the sweep heuristic; "
        "no exact optimum or optimality gap is calculated in this notebook."
    )

    print("\n--- Method characteristics ---")
    print("The angular sort makes route construction systematic and easy to inspect.")
    print("The capacity rule guarantees feasible route loads when each customer")
    print("fits at least one vehicle type, and vehicle selection respects capacity.")
    print("The heuristic does not guarantee globally minimum emissions.")
    print("Its routes depend on customer angles and the chosen depot location.")
    print("The model uses one depot and does not impose fleet-size constraints.")


performance_analysis(problem, routes)

## Relationship to an exact mathematical formulation

The sweep algorithm is a constructive heuristic: it produces routes by angular sorting and capacity-based grouping, then assigns vehicles using their emissions factors. An exact mathematical formulation can represent a broader set of routing decisions, but its computational effort depends on the formulation, solver, and instance. This notebook neither implements nor benchmarks such a formulation, so no numerical comparison or optimality-gap claim is made here.

## Strengths, limitations, and appropriate use

**Strengths**

- Sorting gives a clear, repeatable order for route construction.
- The procedure is straightforward to trace from customer angles to route assignments.
- Capacity checks and vehicle selection are explicit and auditable.
- For a fixed number of vehicle types, the sorting step gives an overall time complexity of \(O(n\log n)\).

**Limitations**

- The heuristic does not guarantee a globally minimum-emissions solution.
- Angular proximity is a geometric grouping rule; it does not account for road networks or travel times.
- The method assumes a single depot and unlimited availability of each vehicle type.
- Route order is fixed by the sweep, and the model omits operational constraints such as time windows and load-dependent emissions.

This approach is useful when a transparent, quickly constructed feasible solution is needed under the stated assumptions. The calculated emissions should be interpreted in the context of those assumptions, rather than as a general performance guarantee. Natural extensions include adding operational constraints or evaluating the same construction rule on larger, explicitly specified instances; those extensions are outside the scope of this implementation.